# BIFlow — Notebook 2 : Analyste BI

**Objectif métier :** où observe-t-on des évolutions inhabituelles du volume de demandes ou des délais de clôture, et quelles zones ou catégories méritent une investigation ?

Le notebook calcule les KPI, tendances, anomalies et preuves avec des règles déterministes. Un seul modèle local, Llama 3.2 3B via Ollama, rédige ensuite un bref résumé qualitatif à partir des insights agrégés. Les chiffres et décisions du rapport restent ceux des calculs vérifiables ; aucune API distante ni clé n’est utilisée.

## Architecture et fichiers nécessaires

Ordre de lecture : chargement/contrôles → descriptif → tendances → anomalies → fiches d’insights → synthèse et exports. Le fichier principal est `data/processed/nyc311_clean.csv`, produit par `notebooks/01_etl_kpi.ipynb`. Le catalogue `data/processed/kpi_catalog.csv` est chargé s’il existe, pour conserver le contexte sémantique. Le notebook ne télécharge aucune donnée.

Chaque section explique l’objectif, son utilité pour l’analyste, l’interprétation attendue et les limites. Les comparaisons mensuelles nécessitent plusieurs mois ; les groupes de délais ont un effectif minimal explicite. Les dates de fermeture, et non le statut textuel, définissent ici une demande clôturée.

## 1. Chargement et contrôles

On charge la sortie nettoyée sans la modifier et on travaille sur une copie analytique. Les contrôles de colonnes optionnelles permettent de désactiver proprement une analyse si une dimension manque. Les dates sont converties dans la copie et les périodes effectivement couvertes sont affichées.

**À interpréter :** une période concentrée sur un mois, beaucoup de demandes sans fermeture, ou des délais absents réduisent fortement la portée des comparaisons. L’absence d’une colonne facultative n’est pas compensée par une colonne inventée.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 120)
plt.style.use("seaborn-v0_8-whitegrid")

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                      if (p / "data/processed/nyc311_clean.csv").exists()), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
CLEAN_PATH = PROJECT_ROOT / "data/processed/nyc311_clean.csv"
CATALOG_PATH = PROJECT_ROOT / "data/processed/kpi_catalog.csv"
SAMPLE_METADATA_PATH = PROJECT_ROOT / "data/raw/nyc311_sample_metadata.json"
SAMPLE_METADATA_PATH = PROJECT_ROOT / "data/raw/nyc311_sample_metadata.json"
SAMPLE_METADATA_PATH = PROJECT_ROOT / "data/raw/nyc311_sample_metadata.json"
SAMPLE_METADATA_PATH = PROJECT_ROOT / "data/raw/nyc311_sample_metadata.json"
SAMPLE_METADATA_PATH = PROJECT_ROOT / "data/raw/nyc311_sample_metadata.json"
SAMPLE_METADATA_PATH = PROJECT_ROOT / "data/raw/nyc311_sample_metadata.json"
if not CLEAN_PATH.exists():
    raise FileNotFoundError(
        f"Fichier requis absent : {CLEAN_PATH}. Exécutez d’abord toutes les cellules "
        "de notebooks/01_etl_kpi.ipynb avec le CSV placé dans data/raw/nyc311_sample.csv."
    )
df_source = pd.read_csv(CLEAN_PATH, low_memory=False)
df = df_source.copy(deep=True)
catalog = pd.read_csv(CATALOG_PATH) if CATALOG_PATH.exists() else None
sample_metadata = json.loads(SAMPLE_METADATA_PATH.read_text(encoding="utf-8")) if SAMPLE_METADATA_PATH.exists() else None
sample_metadata = json.loads(SAMPLE_METADATA_PATH.read_text(encoding="utf-8")) if SAMPLE_METADATA_PATH.exists() else None
sample_metadata = json.loads(SAMPLE_METADATA_PATH.read_text(encoding="utf-8")) if SAMPLE_METADATA_PATH.exists() else None
sample_metadata = json.loads(SAMPLE_METADATA_PATH.read_text(encoding="utf-8")) if SAMPLE_METADATA_PATH.exists() else None
sample_metadata = json.loads(SAMPLE_METADATA_PATH.read_text(encoding="utf-8")) if SAMPLE_METADATA_PATH.exists() else None
sample_metadata = json.loads(SAMPLE_METADATA_PATH.read_text(encoding="utf-8")) if SAMPLE_METADATA_PATH.exists() else None
print("Fichier nettoyé :", CLEAN_PATH, "| dimensions :", df.shape)
print("Catalogue KPI :", "disponible" if catalog is not None else "absent (facultatif)")
print("Métadonnées d’échantillonnage :", "disponibles" if sample_metadata is not None else "absentes")
print("Métadonnées d’échantillonnage :", "disponibles" if sample_metadata is not None else "absentes")
print("Métadonnées d’échantillonnage :", "disponibles" if sample_metadata is not None else "absentes")
print("Métadonnées d’échantillonnage :", "disponibles" if sample_metadata is not None else "absentes")
print("Métadonnées d’échantillonnage :", "disponibles" if sample_metadata is not None else "absentes")
print("Métadonnées d’échantillonnage :", "disponibles" if sample_metadata is not None else "absentes")
display(df.head())
display(df.dtypes.rename("type initial").to_frame())
print("Colonnes réellement disponibles :", df.columns.tolist())

required = {"volume_mensuel": ["created_date"],
            "delais": ["created_date", "closed_date", "resolution_days"],
            "par_type": ["complaint_type"], "par_arrondissement": ["borough"],
            "par_agence": ["agency"]}
availability = pd.DataFrame([{"analyse": name, "disponible": all(c in df.columns for c in cols),
                              "colonnes_manquantes": [c for c in cols if c not in df.columns]}
                             for name, cols in required.items()])
display(availability)
for col in ["created_date", "closed_date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)
if "resolution_days" in df.columns:
    df["resolution_days"] = pd.to_numeric(df["resolution_days"], errors="coerce")
    df.loc[~np.isfinite(df["resolution_days"]), "resolution_days"] = np.nan
    df.loc[df["resolution_days"] < 0, "resolution_days"] = np.nan

if "created_date" in df.columns:
    print("Dates de création valides :", int(df["created_date"].notna().sum()), "/", len(df))
    print("Période couverte :", df["created_date"].min(), "→", df["created_date"].max())
    df["month"] = df["created_date"].dt.to_period("M").astype("string")
else:
    df["month"] = pd.Series(pd.NA, index=df.index, dtype="string")
if "closed_date" in df.columns:
    print("Dates de fermeture valides :", int(df["closed_date"].notna().sum()))
if "resolution_days" in df.columns:
    print("Délais numériques valides :", int(df["resolution_days"].notna().sum()))
print("Limite : les demandes sans date de fermeture peuvent être en cours ; elles ne sont pas traitées comme clôturées.")

## 2. Analyse descriptive

Les ventilations répondent à la question « quoi/où/qui ? ». Elles comptent les lignes de l’échantillon et excluent une dimension manquante uniquement du tableau correspondant. Pour le volume mensuel, les métadonnées de l’extrait fournissent aussi les totaux agrégés officiels du portail ; le notebook les utilise lorsqu’ils concordent avec les mois présents. Les ventilations par catégorie et les délais restent calculés sur l’échantillon de 10 000 lignes. Pour les délais, on limite la population aux demandes ayant une date de fermeture et un délai positif ou nul exploitable ; médiane et quantiles complètent la moyenne.

**À interpréter :** les tableaux et graphiques décrivent l’échantillon. Un volume élevé ne mesure ni le taux de problème par habitant ni la performance de l’agence. Sans dénominateur de population, on ne compare pas les arrondissements comme des taux.

In [ ]:
def volume_table(frame, column):
    if column not in frame.columns:
        print(f"Analyse ignorée : colonne facultative {column!r} absente.")
        return None
    return (frame.loc[frame[column].notna()].groupby(column).size()
            .rename("request_count").sort_values(ascending=False).reset_index())

volume_by_type = volume_table(df, "complaint_type")
volume_by_borough = volume_table(df, "borough")
volume_by_agency = volume_table(df, "agency")
for table in [volume_by_type, volume_by_borough, volume_by_agency]:
    if table is not None:
        display(table.head(15))

monthly_volume = None
if df["month"].notna().any():
    monthly_volume = (df.dropna(subset=["month"]).groupby("month").size()
                      .rename("request_count").reset_index())
    monthly_volume["sample_count"] = monthly_volume["request_count"]
    monthly_volume["count_basis"] = "sample rows"
    if sample_metadata is not None:
        population_counts = sample_metadata.get("population_monthly_counts", {})
        if set(monthly_volume["month"]).issubset(population_counts):
            monthly_volume["request_count"] = monthly_volume["month"].map(population_counts).astype(int)
            monthly_volume["count_basis"] = "official monthly aggregate; sample used for dimensions/delays"
    monthly_volume["month_start"] = pd.PeriodIndex(monthly_volume["month"], freq="M").to_timestamp()
    monthly_volume = monthly_volume.sort_values("month_start").reset_index(drop=True)
    display(monthly_volume)
else:
    print("Volume mensuel non calculable : aucune date de création valide.")

closed_mask = pd.Series(False, index=df.index)
if "closed_date" in df.columns:
    closed_mask = df["closed_date"].notna()
valid_delay_mask = closed_mask & df["resolution_days"].notna() if "resolution_days" in df.columns else closed_mask
valid_delays = df.loc[valid_delay_mask, "resolution_days"].dropna() if "resolution_days" in df.columns else pd.Series(dtype=float)
delay_distribution = pd.DataFrame([{
    "closed_with_valid_delay_n": int(valid_delay_mask.sum()),
    "mean_days": valid_delays.mean() if len(valid_delays) else np.nan,
    "median_days": valid_delays.median() if len(valid_delays) else np.nan,
    "p25_days": valid_delays.quantile(.25) if len(valid_delays) else np.nan,
    "p75_days": valid_delays.quantile(.75) if len(valid_delays) else np.nan,
    "p90_days": valid_delays.quantile(.90) if len(valid_delays) else np.nan,
}])
display(delay_distribution)
print("Moyenne = sensible aux longues durées ; médiane/quantiles décrivent mieux une distribution asymétrique.")

if monthly_volume is not None:
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(monthly_volume["month_start"], monthly_volume["request_count"], marker="o")
    ax.set(title="Volume mensuel officiel" + (" — octobre partiel" if sample_metadata is not None and "2026-10" in sample_metadata.get("partial_months", []) else ""), xlabel="Mois de création", ylabel="Nombre de demandes")
    plt.xticks(rotation=45); plt.tight_layout(); plt.show()
if len(valid_delays):
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(valid_delays, bins=35, color="steelblue")
    ax.set(title="Délais de clôture valides", xlabel="Jours calendaires", ylabel="Nombre de demandes")
    plt.tight_layout(); plt.show()
print("Limite : les demandes sans fermeture/délai valide n’entrent pas dans la distribution des délais.")

## Renforcement statistique — fonctions d'analyse réutilisables

Cette extension conserve les analyses, fonctions, chemins et sorties du notebook existant. Elle ajoute des contrôles préalables, comparaisons et détections explicables, sans recalculer les KPI définis par le Notebook 1. Le catalogue existant fournit notamment les définitions de `monthly_request_volume`, `median_resolution_by_group` et `median_resolution_days`. Le rapport qualité n'est exploité que s'il correspond aux lignes actuellement chargées ; un rapport d'un autre jeu est signalé comme non concordant.

**Méthodes et limites :** le Z-score classique utilise moyenne/écart-type d'un historique antérieur ; le score MAD robuste utilise médiane/MAD. Tous deux s'abstiennent si l'historique est trop court ou la dispersion nulle. Les seuils statistiques servent au repérage et ne sont pas des objectifs métier.

In [ ]:
# Contrôles additionnels sur le schéma réel et les sources de qualité disponibles.
import importlib.util
import hashlib
from datetime import datetime, timezone

def check_columns(frame, columns):
    """Retourne les colonnes requises disponibles et absentes, sans modifier frame."""
    present = [c for c in columns if c in frame.columns]
    missing = [c for c in columns if c not in frame.columns]
    return {"present": present, "missing": missing, "ready": not missing}

def data_quality_precheck(frame):
    """Résume types, manquants, dates, délais et couverture temporelle exploitable."""
    result = {"rows": int(len(frame)), "missing_percent": (frame.isna().mean()*100).round(2).to_dict()}
    for col in ["created_date", "closed_date"]:
        if col in frame:
            parsed = pd.to_datetime(frame[col], errors="coerce", utc=True)
            result[col] = {"valid_n": int(parsed.notna().sum()), "invalid_or_missing_n": int(parsed.isna().sum()),
                           "min": None if parsed.notna().sum()==0 else str(parsed.min()),
                           "max": None if parsed.notna().sum()==0 else str(parsed.max())}
    if "resolution_days" in frame:
        numeric = pd.to_numeric(frame["resolution_days"], errors="coerce")
        result["resolution_days"] = {"valid_finite_nonnegative_n": int((np.isfinite(numeric)&numeric.ge(0)).sum()),
                                     "invalid_or_negative_n": int((~np.isfinite(numeric)|numeric.lt(0)).sum())}
    return result

precheck = data_quality_precheck(df)
print("Contrôle pré-analytique :")
display(pd.DataFrame([{"contrôle": k, "résultat": str(v)} for k,v in precheck.items()]))

quality_candidates = [PROJECT_ROOT/"data/processed/data_quality_report.json",
                      PROJECT_ROOT/"biflow_analyste/data_quality_report.json"]
quality_report_path = next((p for p in quality_candidates if p.exists()), None)
quality_report = None
quality_report_status = "absent"
if quality_report_path is not None:
    loaded_quality = json.loads(quality_report_path.read_text(encoding="utf-8"))
    reported_rows = loaded_quality.get("lignes_propres", loaded_quality.get("rows", loaded_quality.get("row_count")))
    if reported_rows is None or int(reported_rows) == len(df):
        quality_report, quality_report_status = loaded_quality, "disponible_et_concordant"
    else:
        quality_report_status = f"présent_mais_non_concordant ({reported_rows} lignes rapportées vs {len(df)} lignes chargées)"
print("Rapport qualité :", quality_report_status, "|", quality_report_path)

# Lecture de quelques champs bruts pour vérifier les doublons potentiellement à l'origine de pics.
# Lecture seule ; la règle de dédoublonnage du Notebook 1 n'est pas reproduite ici.
RAW_PATH = PROJECT_ROOT/"data/raw/nyc311_sample.csv"
raw_duplicate_by_month = pd.DataFrame(columns=["month", "raw_duplicate_rows"])
raw_quality_status = "source_brute_absente"
if RAW_PATH.exists() and "unique_key" in df.columns and "created_date" in df.columns:
    raw_check = pd.read_csv(RAW_PATH, usecols=lambda c: c in {"unique_key", "created_date"}, low_memory=False)
    raw_check["created_date"] = pd.to_datetime(raw_check["created_date"], errors="coerce", utc=True)
    raw_check["month"] = raw_check["created_date"].dt.to_period("M").astype("string")
    duplicate_mask = raw_check["unique_key"].notna() & raw_check.duplicated("unique_key", keep="first")
    raw_duplicate_by_month = (raw_check.loc[duplicate_mask].groupby("month").size()
                              .rename("raw_duplicate_rows").reset_index())
    raw_quality_status = "contrôle_clé_raw_effectué"
print("Contrôle doublons raw :", raw_quality_status, "| lignes répétées :", int(raw_duplicate_by_month["raw_duplicate_rows"].sum()) if len(raw_duplicate_by_month) else 0)

## 3. Tendances temporelles

Une comparaison de tendance demande une série mensuelle triée et plusieurs périodes complètes. Nous calculons une pente linéaire descriptive du volume et une variation première/dernière période, seulement à partir de trois mois observés. Cette méthode simple évite une décomposition saisonnière inadaptée à une série courte ou irrégulière. Les délais mensuels sont calculés sur les seules demandes fermées avec délai valide.

**À interpréter :** la pente indique une direction moyenne, pas une cause ni une prévision. Un mois partiel à la fin de l’extrait peut artificiellement faire baisser le volume ; la couverture doit être examinée avant toute comparaison.

In [ ]:
def temporal_trend(series, value_col, metric_name, min_periods=3):
    if series is None or len(series) < min_periods:
        return {"metric": metric_name, "n_periods": 0 if series is None else len(series),
                "direction": "insufficient_data", "slope_per_month": np.nan,
                "first_value": np.nan, "last_value": np.nan, "change_percent": np.nan}
    y = series[value_col].astype(float).to_numpy()
    x = np.arange(len(y), dtype=float)
    slope = float(np.polyfit(x, y, 1)[0])
    first, last = float(y[0]), float(y[-1])
    change = (last - first) / abs(first) * 100 if first != 0 else np.nan
    direction = "up" if slope > 0 else "down" if slope < 0 else "flat"
    return {"metric": metric_name, "n_periods": len(series), "direction": direction,
            "slope_per_month": slope, "first_value": first, "last_value": last,
            "change_percent": change}

volume_trend = temporal_trend(monthly_volume, "request_count", "monthly_request_volume")
monthly_delay = None
if "month" in df.columns and "resolution_days" in df.columns:
    delay_population = df.loc[valid_delay_mask & df["month"].notna()]
    if len(delay_population):
        monthly_delay = (delay_population.groupby("month")["resolution_days"]
                         .agg(median_days="median", sample_size="size").reset_index())
        monthly_delay["month_start"] = pd.PeriodIndex(monthly_delay["month"], freq="M").to_timestamp()
        monthly_delay = monthly_delay.sort_values("month_start").reset_index(drop=True)
delay_trend = temporal_trend(monthly_delay, "median_days", "monthly_median_resolution_days")
trend_results = pd.DataFrame([volume_trend, delay_trend])
display(trend_results)
if monthly_delay is not None:
    display(monthly_delay)
print("Observations temporelles : volume =", volume_trend["n_periods"], "mois ; délai =", delay_trend["n_periods"], "mois.")
print("Limite : sans au moins trois mois, aucune direction de tendance n’est affirmée. Contrôler les mois incomplets.")

## Comparaisons de groupes, tendances et anomalies par deux méthodes

Les tableaux comparent des médianes de délai avec leurs effectifs et rangs ; les échantillons sous le minimum sont marqués et non promus comme conclusion. La tendance fournit variations absolues/relatives et moyenne mobile trois périodes. Les anomalies utilisent une période de référence antérieure, afin que la valeur testée ne participe pas à sa propre référence. Z-score et MAD sont calculés sur les mêmes observations. Une MAD ou un écart-type nul produit un statut non calculable, jamais un faux score.

La saisonnalité STL mensuelle exige au minimum deux cycles annuels complets (24 mois), un index mensuel régulier et aucune valeur manquante. Les données ne sont pas interpolées. `statsmodels` est optionnel : son absence ne bloque pas les analyses de base.

In [ ]:
from bi_analyst.analytics import (
    MAD_THRESHOLD, MIN_GROUP_N, MIN_REFERENCE_N, MIN_STL_MONTHS, RULE_VERSION,
    Z_THRESHOLD, compare_group_delays, compare_to_target, detect_mad, detect_zscore,
    monthly_trend_table, persistent_trend, run_stl, scan_group_month_anomalies,
    scan_monthly_anomalies, stl_residual_signals,
)

group_comparisons = compare_group_delays(df, ['borough', 'complaint_type', 'agency'], catalog=catalog)

partial_months = set(sample_metadata.get('partial_months', [])) if sample_metadata is not None else set()

trend_monthly_volume = monthly_volume.loc[~monthly_volume['month'].isin(partial_months)].copy() if monthly_volume is not None else None

if monthly_volume is not None and len(partial_months):
    print('Mois partiels exclus des tendances :', sorted(partial_months))

trend_analysis = monthly_trend_table(trend_monthly_volume)

volume_trend = temporal_trend(trend_monthly_volume, 'request_count', 'monthly_request_volume')

persistent_volume_trend = persistent_trend(trend_analysis)

trend_monthly_delay = monthly_delay.loc[~monthly_delay['month'].isin(partial_months)].copy() if monthly_delay is not None else None

delay_trend = temporal_trend(trend_monthly_delay, 'median_days', 'monthly_median_resolution_days')

trend_results = pd.DataFrame([volume_trend, delay_trend])

monthly_series = trend_monthly_volume.set_index('month')['request_count'] if trend_monthly_volume is not None else None

if monthly_series is not None:
    monthly_series.index = pd.PeriodIndex(monthly_series.index, freq='M')

stl_result = run_stl(monthly_series, period=12)

stl_decomposition = stl_result['decomposition']

stl_residual_comparison = stl_residual_signals(stl_decomposition)

median_resolution = float(delay_distribution.loc[0, 'median_days']) if pd.notna(delay_distribution.loc[0, 'median_days']) else np.nan

target_check = compare_to_target(median_resolution, None, direction='lower_is_better', period='période du fichier', source=None)

anomaly_comparison = scan_monthly_anomalies(trend_monthly_volume)

group_anomaly_parts = [scan_group_month_anomalies(df, c, partial_months=partial_months) for c in ['complaint_type', 'borough', 'agency'] if c in df]

group_anomaly_parts = [x for x in group_anomaly_parts if len(x)]

if group_anomaly_parts:
    anomaly_comparison = pd.concat([anomaly_comparison, *group_anomaly_parts], ignore_index=True)

advanced_anomalies = anomaly_comparison.loc[anomaly_comparison['status'] == 'anomaly'].copy() if len(anomaly_comparison) else anomaly_comparison.copy()

if len(stl_residual_comparison):
    stl_residual_anomalies = stl_residual_comparison.loc[stl_residual_comparison['status'] == 'anomaly'].copy()
    if len(stl_residual_anomalies):
        advanced_anomalies = pd.concat([advanced_anomalies, stl_residual_anomalies], ignore_index=True, sort=False)

print('Comparaisons de groupes :', len(group_comparisons), '| mois pour tendance :', len(trend_analysis), '| STL :', stl_result['status'], stl_result['reason'], '| cible :', target_check['status'])

display(group_comparisons.head(20))

display(trend_analysis)

display(anomaly_comparison)

display(advanced_anomalies)


## 4. Anomalies de volume et délais relativement élevés

Pour une série mensuelle suffisamment longue, la référence utilise le score robuste `0,6745 × (volume − médiane) / MAD`, avec seuil absolu de **3,5**. Il faut au moins six mois. Si le MAD est nul (série très stable), un repli explicite signale les écarts d’au moins 20 % de la médiane. Ce seuil de repli sert à détecter des écarts à investiguer ; ce n’est pas un seuil de qualité métier. Le calcul s’applique au total et, si disponibles, aux catégories/arrondissements observés sur plusieurs mois.

Pour les délais, les groupes sont comparés à la distribution des médianes de leurs pairs ; seuls les groupes ayant au moins **30** délais valides sont classés. Le 75e percentile indique des délais relativement élevés, sans SLA ni jugement de non-performance. Vérifier les anomalies contre la couverture, les doublons et les changements de collecte ; ces causes ne peuvent pas être établies par ce seul extrait.

In [ ]:
from bi_analyst.analytics import (
    MIN_GROUP_N, MIN_MONTHS_ANOMALY, ROBUST_Z_THRESHOLD,
    ZERO_MAD_RELATIVE_THRESHOLD, delay_groups, robust_month_anomalies,
)
MIN_DELAY_GROUP_N = MIN_GROUP_N

anomaly_records = robust_month_anomalies(df, partial_months=partial_months)

for col in ['complaint_type', 'borough']:
    if col in df.columns:
        anomaly_records.extend(robust_month_anomalies(df, col, col, partial_months=partial_months))

anomaly_columns = ['dimension', 'group', 'period', 'volume', 'baseline_median', 'mad', 'score', 'method', 'threshold', 'n_months']

anomalies = pd.DataFrame(anomaly_records, columns=anomaly_columns)

print('Anomalies détectées :', len(anomalies), '| règle : MAD robuste seuil', ROBUST_Z_THRESHOLD, '| minimum', MIN_MONTHS_ANOMALY, 'mois')

display(anomalies.sort_values('score', key=lambda s: s.abs(), ascending=False) if len(anomalies) else anomalies)

delay_by_type = delay_groups(df, 'complaint_type')

delay_by_borough = delay_groups(df, 'borough')

high_delay_groups = []

for dim, table in [('complaint_type', delay_by_type), ('borough', delay_by_borough)]:
    if table is None:
        print(f'Délai par {dim} indisponible : colonne ou délai absent.')
        continue
    if len(table):
        cutoff = float(table['median_days'].quantile(0.75))
        for _, row in table.loc[table['median_days'] >= cutoff].iterrows():
            high_delay_groups.append({'dimension': dim, 'group': str(row[dim]), 'median_days': float(row['median_days']), 'sample_size': int(row['sample_size']), 'peer_p75_days': cutoff, 'minimum_group_n': MIN_DELAY_GROUP_N})
        display(table.head(10))

high_delay = pd.DataFrame(high_delay_groups, columns=['dimension', 'group', 'median_days', 'sample_size', 'peer_p75_days', 'minimum_group_n'])

print('Groupes aux médianes dans le quart supérieur de leurs groupes admissibles (comparaison descriptive, aucun SLA) :')

display(high_delay)

print('À vérifier : un pic peut refléter un mois partiel, un changement de collecte ou des répétitions ; le CSV seul ne tranche pas la cause.')


## 5. Génération déterministe des fiches d’insights

Une fiche sépare le fait observé, son interprétation possible et l’action à envisager. Elle est générée uniquement à partir des tendances, anomalies et médianes calculées ci-dessus. La confiance est une appréciation de la quantité de données (périodes/effectifs), pas une probabilité. Une recommandation invite à vérifier ou investiguer ; elle n’attribue pas de cause.

In [ ]:
def confidence_for(n, periods=0):
    if periods >= 12 and n >= 100: return "medium"
    if periods >= 6 and n >= 30: return "medium"
    return "low"

def make_insight(insight_id, insight_type, period, dimension, observation, evidence,
                 interpretation, recommendation, confidence, limitations):
    return {"insight_id": insight_id, "insight_type": insight_type, "period": period,
            "dimension": dimension, "observation": observation, "evidence": evidence,
            "interpretation": interpretation, "recommendation": recommendation,
            "confidence": confidence, "limitations": limitations}

insights = []
if volume_trend["direction"] in ["up", "down"]:
    insights.append(make_insight(
        f"trend-volume-{len(insights)+1:03d}", "tendance", "période observée", "volume total",
        f"Le volume mensuel suit une tendance {('à la hausse' if volume_trend['direction']=='up' else 'à la baisse')}.",
        {k: (None if pd.isna(v) else v) for k,v in volume_trend.items()},
        "La pente résume la période observée ; elle ne permet pas d’identifier une cause.",
        "Vérifier les mois de début et de fin, puis comparer avec les événements de service et la couverture de collecte.",
        confidence_for(volume_trend["n_periods"], volume_trend["n_periods"]),
        "Tendance descriptive linéaire ; série courte ou mois partiel possibles."))
for _, row in anomalies.iterrows():
    evidence = {k: (None if pd.isna(v) else v.item() if isinstance(v, np.generic) else v) for k,v in row.to_dict().items()}
    insights.append(make_insight(
        f"anomaly-{len(insights)+1:03d}", "anomalie", str(row["period"]), f"{row['dimension']}={row['group']}",
        f"Volume atypique détecté : {int(row['volume'])} demandes, référence médiane {row['baseline_median']:.1f}.",
        evidence, "Écart statistique à examiner ; il ne prouve pas une anomalie opérationnelle.",
        "Contrôler l’exhaustivité du mois, les doublons et les changements de collecte avant d’alerter l’équipe.",
        confidence_for(int(row["volume"]), int(row["n_months"])),
        "Score robuste exploratoire ; cause non identifiable dans ces champs."))
for item in high_delay_groups:
    insights.append(make_insight(
        f"delay-{len(insights)+1:03d}", "délai élevé", "période couverte", f"{item['dimension']}={item['group']}",
        f"Médiane de clôture {item['median_days']:.1f} jours parmi {item['sample_size']} demandes valides.",
        item, "Cette médiane se situe dans le quart supérieur des groupes comparables admissibles.",
        "Examiner un échantillon de dossiers et les étapes de traitement pour comprendre les délais ; aucun SLA n’est fourni.",
        confidence_for(item["sample_size"]),
        "Comparaison descriptive ; différences de composition des demandes possibles, aucune causalité établie."))

insight_columns = ["insight_id", "insight_type", "period", "dimension", "observation", "evidence",
                   "interpretation", "recommendation", "confidence", "limitations"]
insights_df = pd.DataFrame(insights, columns=insight_columns)
if len(insights_df):
    insights_df["evidence"] = insights_df["evidence"].apply(lambda x: json.dumps(x, ensure_ascii=False, default=str))
display(insights_df)
print("Nombre de fiches construites à partir des résultats mesurés :", len(insights_df))

## Insights enrichis, preuves et auditabilité

Les fiches ci-dessous combinent uniquement résultats mesurés et règles versionnées. Chaque fiche contient KPI/définition, valeurs de référence, score, seuil, période, règle déclenchée, limites, priorité calculée par écart relatif (≥50 %), effectif (≥100) et signal de doublons, ainsi qu'un état de qualité. La priorité (1–3) est un rang de tri déterministe, pas une probabilité. La source d'un insight reste retrouvable via `evidence_id` dans le tableau de preuves agrégées.

In [ ]:
def build_auditable_insights(anomaly_rows, group_delays, trend_state, quality_duplicates, target_state):
    """Construit des insights à règles fixes et un registre détaillé de preuves."""
    records=[]; evidence_rows=[]
    volume_def=(catalog.loc[catalog["kpi_name"]=="monthly_request_volume","business_definition"].iloc[0]
                if catalog is not None and (catalog["kpi_name"]=="monthly_request_volume").any()
                else "Définition KPI absente du catalogue ; volume agrégé par lignes datées")
    def add(kind,period,dimension,group,observation,evidence,method,threshold,reference_period,interpretation,recommendation,n,quality,limitations,rule):
        rel=abs(float(evidence.get("relative_deviation",0) or 0))
        quality_text=str(quality)
        quality_risk=("doublons_raw_période=" in quality_text and not quality_text.endswith("=0"))
        # Priorité de tri (1–3), pas une probabilité : ampleur ≥50 %, effectif ≥100, signal qualité.
        priority=min(3,1+int(rel>=.5)+int(n>=100)+int(quality_risk))
        iid=f"BI-{len(records)+1:04d}"
        eid=f"EV-{len(evidence_rows)+1:04d}"
        evidence_full={**evidence,"evidence_id":eid,"rule_id":rule,"rule_version":RULE_VERSION,
                      "kpi_definition":volume_def if kind in ("hausse inhabituelle","baisse inhabituelle","tendance persistante","données insuffisantes") else "Médiane de resolution_days sur demandes clôturées exploitables ; définition selon catalogue KPI."}
        records.append({"insight_id":iid,"insight_type":kind,"period":period,"dimension":dimension,"group_value":group,
                        "observation":observation,"evidence":json.dumps(evidence_full,ensure_ascii=False,default=str),"method":method,
                        "threshold":threshold,"reference_period":reference_period,"interpretation":interpretation,
                        "recommendation":recommendation,"priority":priority,"confidence":"medium" if n>=100 else "low",
                        "limitations":limitations,"data_quality_status":quality,"evidence_id":eid,"rule_id":rule})
        evidence_rows.append({"evidence_id":eid,"insight_id":iid,"period":period,"dimension":dimension,"group_value":group,
                              "observed":evidence.get("observed"),"reference":evidence.get("reference"),"score":evidence.get("score"),
                              "threshold":threshold,"method":method,"reference_period":reference_period,
                              "kpi_definition":evidence_full["kpi_definition"],"rule_id":rule,"rule_version":RULE_VERSION,
                              "aggregated_values":json.dumps(evidence.get("aggregated_values",{}),ensure_ascii=False,default=str),
                              "limitations":limitations})
    if trend_state["status"]=="persistent":
        add("tendance persistante",f"{trend_state['start_period']}–{trend_state['end_period']}","volume mensuel","all",
            f"Tendance persistante {trend_state['direction']} sur les dernières périodes.",
            {"observed":trend_state["end_value"],"reference":trend_state["start_value"],"relative_deviation":trend_state["change_pct"],"aggregated_values":trend_state},
            "variations mensuelles successives",trend_state["required_same_direction"],trend_state["start_period"],
            "Plusieurs variations successives vont dans le même sens ; aucune cause n'est établie.",
            "Vérifier la couverture des périodes et examiner les catégories/zones qui contribuent au changement.",len(trend_analysis),
            raw_quality_status,"Tendance descriptive ; mois incomplets et saisonnalité peuvent influencer la série.","RULE_TREND_PERSISTENT")
    elif trend_state["status"]=="insufficient_periods":
        add("données insuffisantes",None,"volume mensuel","all",
            f"Pas de tendance persistante conclue : {trend_state['status']}.",
            {"observed":None,"reference":None,"aggregated_values":{"period_count":trend_state.get("n_periods",0)}},
            "contrôle de longueur/persistance",4,None,"Les données ne satisfont pas la règle de persistance.",
            "Charger davantage de périodes mensuelles complètes avant d'interpréter une tendance.",int(trend_state.get("n_periods",0)),
            raw_quality_status,"Historique trop court ou changements mensuels non persistants.","RULE_INSUFFICIENT_HISTORY")
    for _,r in anomaly_rows.iterrows():
        direction=r["direction"]
        kind="hausse inhabituelle" if direction=="positive" else "baisse inhabituelle"
        period=r.get("period"); duplicate_n=0
        if len(quality_duplicates) and "month" in quality_duplicates:
            matched=quality_duplicates.loc[quality_duplicates["month"].astype(str)==str(period),"raw_duplicate_rows"]
            duplicate_n=int(matched.sum()) if len(matched) else 0
        qstatus=f"anomalie_statistique; doublons_raw_période={duplicate_n}"
        quality_note="Doublons observés dans le brut sur cette période ; pic potentiellement lié à la qualité, vérifier les clés." if duplicate_n else "Aucun doublon par clé observé dans le contrôle disponible ; autres changements de collecte non vérifiés."
        add(kind,period,r.get("dimension","all"),r.get("group_value","all"),
            f"Valeur {r['observed']:.1f}, score {r['score']:.2f} ({direction}).",
            {"observed":r["observed"],"reference":r.get("reference_mean",r.get("reference_median")),
             "score":r["score"],"relative_deviation":(r["observed"]-r.get("reference_mean",r.get("reference_median")))/max(abs(r.get("reference_mean",r.get("reference_median"))),1e-9),
             "aggregated_values":{"reference_n":int(r["reference_n"]),"reference_std":r.get("reference_std"),"mad":r.get("mad"),"raw_duplicate_rows":duplicate_n}},
            r["method"],r["threshold"],r["reference_period"],
            "Écart statistique à examiner ; il ne prouve ni erreur ni cause opérationnelle.",
            "Vérifier doublons, catégories, zones et couverture de collecte avant de conclure.",int(r["reference_n"]),qstatus,
            quality_note,"RULE_VOLUME_ANOMALY")
    if not group_delays.empty:
        for dim,part in group_delays[group_delays["sample_status"]=="meets_minimum"].groupby("dimension"):
            cutoff=float(part["median_days"].quantile(.75))
            for _,r in part.loc[part["median_days"]>=cutoff].iterrows():
                add("délai relativement élevé",None,dim,r["group_value"],
                    f"Médiane {r['median_days']:.3f} jours (n={int(r['sample_size'])}), quart supérieur comparatif.",
                    {"observed":float(r["median_days"]),"reference":cutoff,"relative_deviation":(float(r["median_days"])-cutoff)/max(abs(cutoff),1e-9),
                     "aggregated_values":{"sample_size":int(r["sample_size"]),"mean_days":float(r["mean_days"]),"p75_days":float(r["p75_days"])}},
                    "médiane de groupe comparée au P75 des groupes admissibles",None,"groupes de même dimension",
                    "Délai relativement élevé parmi les groupes comparables admissibles, pas un dépassement SLA.",
                    "Examiner des dossiers et le parcours de traitement ; valider les différences de composition.",int(r["sample_size"]),
                    raw_quality_status,"Comparaison descriptive ; aucun seuil officiel de délai n'est configuré.","RULE_GROUP_DELAY_P75")
    if target_state["status"]=="cible métier non configurée":
        # Conserver le statut d'absence de cible dans le registre, sans créer un constat de dépassement.
        pass
    return pd.DataFrame(records),pd.DataFrame(evidence_rows)

rule_insights, evidence_registry=build_auditable_insights(advanced_anomalies,group_comparisons,
                                                         persistent_volume_trend,raw_duplicate_by_month,target_check)
print("Insights auditables :",len(rule_insights),"| preuves traçables :",len(evidence_registry),"| version :",RULE_VERSION)
display(rule_insights); display(evidence_registry.head(10))

## 6. Synthèse destinée au responsable de service

La synthèse reformule uniquement les résultats disponibles en langage courant. Les recommandations sont limitées aux investigations que les données justifient. Si la série est trop courte ou qu’aucune anomalie ne franchit le seuil, le rapport le dit explicitement.

In [ ]:
def fmt_num(value, decimals=1):
    return "non calculable" if pd.isna(value) else f"{value:,.{decimals}f}".replace(",", " ")

report_lines = ["# Rapport Analyste BI — NYC 311", "",
                f"Échantillon analysé : {len(df):,} lignes."]
if "created_date" in df:
    report_lines.append(f"Période disponible : {df['created_date'].min()} à {df['created_date'].max()}.")
report_lines += ["", "## Tendances", ""]
if volume_trend["direction"] == "insufficient_data":
    report_lines.append(f"Pas assez de mois observés pour conclure sur une tendance de volume ({volume_trend['n_periods']} mois ; minimum 3).")
else:
    report_lines.append(f"Volume : tendance {('à la hausse' if volume_trend['direction']=='up' else 'à la baisse' if volume_trend['direction']=='down' else 'stable')}, pente descriptive {fmt_num(volume_trend['slope_per_month'])} demandes/mois sur {volume_trend['n_periods']} mois.")
if delay_trend["direction"] == "insufficient_data":
    report_lines.append(f"Délais mensuels : pas assez de périodes valides pour comparer ({delay_trend['n_periods']} mois).")
else:
    report_lines.append(f"Médiane des délais : tendance {delay_trend['direction']} sur {delay_trend['n_periods']} mois.")
report_lines += ["", "## Anomalies à examiner", ""]
if anomalies.empty:
    report_lines.append(f"Détection mensuelle non exécutée : {len(monthly_volume) if monthly_volume is not None else 0} mois disponibles, alors qu’il faut au moins 7 pour comparer une période à son historique. Aucune conclusion d’absence d’anomalie n’est possible.")
else:
    for _, r in anomalies.head(5).iterrows():
        report_lines.append(f"- {r['period']} — {r['dimension']} {r['group']} : {int(r['volume'])} demandes, médiane de référence {fmt_num(r['baseline_median'])}.")
report_lines += ["", "## Délais relativement élevés", ""]
if high_delay.empty:
    report_lines.append(f"Aucun groupe avec au moins {MIN_DELAY_GROUP_N} délais valides n’a été identifié dans le quart supérieur, ou les données sont insuffisantes.")
else:
    for item in high_delay_groups[:5]:
        report_lines.append(f"- {item['dimension']} {item['group']} : médiane {fmt_num(item['median_days'])} jours (n={item['sample_size']}, seuil comparatif P75={fmt_num(item['peer_p75_days'])} jours). Ce n’est pas un constat de non-respect d’un SLA.")
report_lines += ["", "## Recommandations prioritaires", ""]
recommendations = []
if not anomalies.empty:
    recommendations.append("Vérifier la couverture, les doublons et les changements de collecte pour les périodes signalées avant toute escalade.")
if not high_delay.empty:
    recommendations.append("Examiner un échantillon de dossiers dans les groupes aux délais relativement élevés afin d’identifier les étapes à investiguer.")
if volume_trend["direction"] != "insufficient_data" or delay_trend["direction"] != "insufficient_data":
    recommendations.append("Confirmer que les mois comparés sont complets et comparables, surtout aux bornes de l’extrait.")
if recommendations:
    report_lines.extend([f"{i}. {r}" for i,r in enumerate(recommendations[:3],1)])
else:
    report_lines.append("Les données actuelles ne justifient pas de recommandation opérationnelle spécifique ; obtenir une période plus longue et vérifier la couverture.")
report_lines += ["", "## Limites et vérifications", "",
                 "- Les résultats décrivent l’extrait fourni, pas nécessairement toutes les demandes de New York.",
                 "- Les demandes sans date de fermeture sont exclues des délais ; elles peuvent être en cours.",
                 "- Une anomalie statistique ne prouve ni erreur ni cause. Contrôler qualité et contexte opérationnel.",
                 "- Aucun objectif de délai/SLA n’a été fourni ; les délais sont comparés entre groupes admissibles seulement.",
                 "- Les effectifs faibles, mois incomplets et valeurs manquantes limitent les conclusions."]
report_markdown = "\n".join(report_lines)
print(report_markdown)

## 7. Tests, exports et limites

Les assertions vérifient l’ordre des périodes, les effectifs et les seuils effectivement employés, l’absence de valeurs infinies, la correspondance des fiches et l’exclusion des demandes ouvertes des délais. Un test synthétique isolé vérifie qu’un pic artificiel dans une série stable est détecté ; il n’est jamais mélangé aux données NYC 311.

Les fichiers de sortie sont créés dans `outputs/bi_analyst/`. Une table d’anomalies vide garde son schéma afin d’être consommable par les étapes suivantes. Ces contrôles vérifient la logique de référence, pas l’exhaustivité du jeu source ni une causalité métier.

In [ ]:
# Test synthétique séparé : 11 mois stables et un pic artificiel au 12e mois.
# Série indépendante des données réelles, avec 100 demandes/mois puis un pic de 300.
synthetic = pd.DataFrame({"month": sum(([f"2025-{m:02d}"] * (300 if m == 12 else 100) for m in range(1,13)), [])})
synthetic_result = robust_month_anomalies(synthetic)
assert any(r["period"] == "2025-12" for r in synthetic_result), "Le test synthétique n’a pas détecté le pic artificiel."

# Vérifications sur les données réelles.
assert volume_table(df.drop(columns=["agency"], errors="ignore"), "missing_optional_column") is None
assert delay_groups(df.drop(columns=["complaint_type"], errors="ignore"), "complaint_type") is None
assert np.isfinite(trend_results[["slope_per_month", "first_value", "last_value"]].to_numpy(dtype=float)[np.isfinite(trend_results[["slope_per_month", "first_value", "last_value"]].to_numpy(dtype=float))]).all()
if monthly_volume is not None:
    assert monthly_volume["month_start"].is_monotonic_increasing
    assert int(monthly_volume["sample_count"].sum()) == int(df["month"].notna().sum())
if not anomalies.empty:
    robust_rows = anomalies[anomalies["method"] == "robust_z_mad"]
    assert (robust_rows["score"].abs() >= ROBUST_Z_THRESHOLD).all()
    assert (anomalies["n_months"] >= MIN_MONTHS_ANOMALY).all()
if "resolution_days" in df and "closed_date" in df:
    assert df.loc[~closed_mask, "resolution_days"].isna().all(), "Une demande sans fermeture a un délai utilisé."
    assert (df["resolution_days"].dropna() >= 0).all()
if len(insights_df):
    assert insights_df["insight_id"].is_unique
    assert set(insights_df["insight_type"]).issubset({"tendance", "anomalie", "délai élevé"})
assert len(synthetic_result) > 0
print("Contrôles réussis : période triée, effectifs cohérents, scores au seuil, délais limités aux demandes clôturées, insight IDs uniques, pic synthétique détecté.")

OUTPUT_DIR = PROJECT_ROOT / "outputs/bi_analyst"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
insights_path = OUTPUT_DIR / "insights.csv"
anomalies_path = OUTPUT_DIR / "anomalies.csv"
report_path = OUTPUT_DIR / "bi_analyst_report.md"
summary_path = OUTPUT_DIR / "analysis_summary.json"
insights_df.to_csv(insights_path, index=False)
anomalies.to_csv(anomalies_path, index=False)
report_path.write_text(report_markdown, encoding="utf-8")
summary = {
    "input_file": str(CLEAN_PATH), "rows": int(len(df)),
    "created_period_start": None if df["created_date"].notna().sum() == 0 else str(df["created_date"].min()),
    "created_period_end": None if df["created_date"].notna().sum() == 0 else str(df["created_date"].max()),
    "closed_with_valid_delay_n": int(valid_delay_mask.sum()),
    "monthly_volume_periods": int(volume_trend["n_periods"]),
    "anomaly_count": int(len(anomalies)), "insight_count": int(len(insights_df)),
    "anomaly_method": {"min_months": MIN_MONTHS_ANOMALY, "robust_z_threshold": ROBUST_Z_THRESHOLD,
                       "zero_mad_relative_threshold": ZERO_MAD_RELATIVE_THRESHOLD},
    "limitations": ["Échantillon uniquement", "Causes non établies", "Aucun SLA fourni", "Mois partiels possibles"]
}
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

# Relecture des exports : les lignes exportées doivent correspondre aux résultats calculés.
assert len(pd.read_csv(insights_path)) == len(insights_df)
assert len(pd.read_csv(anomalies_path)) == len(anomalies)
assert pd.read_csv(anomalies_path).columns.tolist() == anomaly_columns
assert json.loads(summary_path.read_text(encoding="utf-8"))["anomaly_count"] == len(anomalies)
print("Exports vérifiés :")
for path in [insights_path, anomalies_path, report_path, summary_path]:
    print(f"- {path} ({path.stat().st_size:,} octets)")

## Évaluation synthétique isolée et reproductibilité

Les scénarios artificiels sont créés en mémoire, ne touchent pas `df`, au CSV nettoyé ni aux exports métier existants. Ils évaluent hausse, baisse, absence d'anomalie, doublons contrôlés, historique court, trous/NaN, série constante, dates irrégulières, STL ignoré, reproductibilité et intégrité des sources réelles. Les seuils Z=3 et MAD=3,5 sont testés explicitement.

In [ ]:
# Tests synthétiques séparés — assertions effectives, aucun fichier NYC n'est modifié.
test_results=[]
clean_hash_before_tests=hashlib.sha256(CLEAN_PATH.read_bytes()).hexdigest()
real_export_paths=[insights_path,anomalies_path,report_path,summary_path]
real_export_hashes={str(path):hashlib.sha256(path.read_bytes()).hexdigest() for path in real_export_paths if path.exists()}
def record_test(name,passed,detail):
    test_results.append({"test":name,"passed":bool(passed),"detail":str(detail)})

base=np.array([100,102,99,101,98,103,100,99,101,102,98,100],dtype=float)
z_spike=detect_zscore(base,200,period="synthetic-spike")
mad_spike=detect_mad(base,200,period="synthetic-spike")
assert z_spike["status"]=="anomaly" and z_spike["direction"]=="positive"
assert mad_spike["status"]=="anomaly" and mad_spike["direction"]=="positive"
record_test("pic_artificiel",True,{"period":z_spike["period"],"value":z_spike["observed"],"z":z_spike["score"],"robust_z":mad_spike["score"]})

z_drop=detect_zscore(base,20,period="synthetic-drop"); mad_drop=detect_mad(base,20,period="synthetic-drop")
assert z_drop["status"]=="anomaly" and z_drop["direction"]=="negative"
assert mad_drop["status"]=="anomaly" and mad_drop["direction"]=="negative"
record_test("baisse_artificielle",True,{"period":z_drop["period"],"value":z_drop["observed"],"direction":z_drop["direction"]})

normal_z=detect_zscore(base,101); normal_mad=detect_mad(base,101)
assert normal_z["status"]=="normal" and normal_mad["status"]=="normal"
record_test("serie_sans_anomalie",True,"aucune méthode n'a signalé la valeur ordinaire")

# Pic artificiel dû à copies de clés : avant déduplication l'anomalie Z existe, après déduplication elle disparaît.
synthetic_rows=[]; key_index=0
for month_number,month_volume in enumerate(base.astype(int),1):
    for _ in range(int(month_volume)):
        synthetic_rows.append({"unique_key":f"k{key_index}","month":f"2025-{month_number:02d}"}); key_index+=1
synthetic_unique=pd.DataFrame(synthetic_rows)
synthetic_duplicates=pd.concat([synthetic_unique,synthetic_unique.loc[synthetic_unique["month"]=="2025-12"].iloc[:100]],ignore_index=True)
counts_before=synthetic_duplicates.groupby("month").size()
counts_after=synthetic_duplicates.drop_duplicates("unique_key").groupby("month").size()
duplicate_peak=detect_zscore(counts_before.iloc[:-1],counts_before.iloc[-1],period="2025-12")
dedup_peak=detect_zscore(counts_after.iloc[:-1],counts_after.iloc[-1],period="2025-12")
assert duplicate_peak["status"]=="anomaly" and dedup_peak["status"]=="normal"
duplicate_test_evidence={"before_rows":int(len(synthetic_duplicates)),"after_unique_rows":int(len(synthetic_duplicates.drop_duplicates("unique_key"))),
                         "duplicate_rows":int(len(synthetic_duplicates)-len(synthetic_unique)),"z_before":duplicate_peak["score"],"z_after":dedup_peak["score"]}
record_test("pic_apparent_doublons",True,duplicate_test_evidence)
synthetic_anomaly_row=pd.DataFrame([{"status":"anomaly","direction":"positive","period":"2025-12",
    "dimension":"all","group_value":"all","observed":float(counts_before.iloc[-1]),
    "score":float(duplicate_peak["score"]),"reference_mean":float(duplicate_peak["reference_mean"]),
    "reference_n":int(duplicate_peak["reference_n"]),"reference_period":duplicate_peak["reference_period"],
    "method":"zscore","threshold":Z_THRESHOLD}])
synthetic_quality=pd.DataFrame([{"month":"2025-12","raw_duplicate_rows":duplicate_test_evidence["duplicate_rows"]}])
quality_insights,_=build_auditable_insights(synthetic_anomaly_row,pd.DataFrame(),
    {"status":"persistent","direction":"up","start_period":"2025-01","end_period":"2025-12",
     "start_value":100.0,"end_value":200.0,"change_pct":100.0,"change_abs":100.0,"required_same_direction":3,"n_periods":12},
    synthetic_quality,target_check)
assert quality_insights["data_quality_status"].str.contains("doublons_raw_période=100").any()
assert quality_insights["limitations"].str.contains("Doublons observés").any()
record_test("signal_qualite_doublons",True,"insight étiqueté comme potentiellement lié aux doublons, sans conclure à une cause réelle")

short_z=detect_zscore([1,2,3],20); short_mad=detect_mad([1,2,3],20)
constant_z=detect_zscore([5]*8,20); constant_mad=detect_mad([5]*8,20)
nan_z=detect_zscore([1,np.nan,2,3,4,5,6,7],30)
assert short_z["status"]=="insufficient_history_or_invalid_value" and short_mad["status"]=="insufficient_history_or_invalid_value"
assert constant_z["status"]=="zero_or_invalid_std" and constant_mad["status"]=="zero_or_invalid_mad"
assert nan_z["reference_n"]==7
record_test("court_nan_constant",True,{"short":short_z["status"],"constant_z":constant_z["status"],"constant_mad":constant_mad["status"],"nan_reference_n":nan_z["reference_n"]})

irregular_index=pd.period_range("2022-01",periods=25,freq="M").delete(5)
irregular=pd.Series(np.linspace(100,125,len(irregular_index)),index=irregular_index)
stl_irregular=run_stl(irregular,period=12)
stl_short=run_stl(pd.Series([1,2,3],index=pd.period_range("2024-01",periods=3,freq="M")),period=12)
assert stl_irregular["status"]=="skipped" and "irrégulier" in stl_irregular["reason"]
assert stl_short["status"]=="skipped" and stl_short["reason"] is not None
record_test("stl_historique_irrégulier_court",True,{"irregular":stl_irregular["reason"],"short":stl_short["reason"]})
regular_index=pd.period_range("2021-01",periods=36,freq="M")
regular_values=pd.Series(100+np.arange(36)*.4+8*np.sin(2*np.pi*np.arange(36)/12),index=regular_index)
regular_stl=run_stl(regular_values,period=12)
if importlib.util.find_spec("statsmodels") is not None:
    assert regular_stl["status"]=="executed" and set(["trend","seasonal","resid"]).issubset(regular_stl["decomposition"].columns)
record_test("stl_serie_reguliere",regular_stl["status"]=="executed" if importlib.util.find_spec("statsmodels") is not None else regular_stl["status"]=="skipped",regular_stl["reason"] or "décomposition synthétique produite")
residual_values=np.array([0.1,-0.1,0.05,-0.05,0.0,0.08,-0.08,0.02,-0.02,0.04,-0.04,0.0,2.0])
synthetic_decomposition=pd.DataFrame({"resid":residual_values},index=pd.period_range("2024-01",periods=len(residual_values),freq="M"))
synthetic_residual_flags=stl_residual_signals(synthetic_decomposition)
assert synthetic_residual_flags.iloc[-1]["status"]=="anomaly"
record_test("signal_résidu_STL_synthétique",True,{"period":synthetic_residual_flags.iloc[-1]["period"],"score":synthetic_residual_flags.iloc[-1]["score"]})

regular_trend=pd.DataFrame({"month":[f"2025-{m:02d}" for m in range(1,5)],
    "month_start":pd.date_range("2025-01-01",periods=4,freq="MS"),"request_count":[100,110,120,130]})
assert persistent_trend(regular_trend)["status"]=="persistent" and persistent_trend(regular_trend)["direction"]=="up"
isolated_trend=pd.DataFrame({"month":[f"2025-{m:02d}" for m in range(1,5)],
    "month_start":pd.date_range("2025-01-01",periods=4,freq="MS"),"request_count":[100,100,150,100]})
assert persistent_trend(isolated_trend)["status"]=="no_persistent_direction"
record_test("tendance_persistante_vs_isolee",True,"trois hausses successives reconnues ; variation isolée non déclarée persistante")

# Reproductibilité : mêmes valeurs, paramètres et ordre des règles produisent les mêmes résultats.
rep1=detect_mad(base,200,period="2025-12"); rep2=detect_mad(base.copy(),200,period="2025-12")
assert rep1==rep2
ins1,ev1=build_auditable_insights(advanced_anomalies,group_comparisons,persistent_volume_trend,raw_duplicate_by_month,target_check)
ins2,ev2=build_auditable_insights(advanced_anomalies,group_comparisons,persistent_volume_trend,raw_duplicate_by_month,target_check)
pd.testing.assert_frame_equal(ins1,ins2); pd.testing.assert_frame_equal(ev1,ev2)
record_test("reproductibilite",True,"scores, règles, preuves et observations identiques")

# Aucun test synthétique ne modifie les objets métier ni le fichier nettoyé.
assert len(df)==len(df_source) and df_source.equals(pd.read_csv(CLEAN_PATH,low_memory=False))
assert hashlib.sha256(CLEAN_PATH.read_bytes()).hexdigest()==clean_hash_before_tests
assert all(hashlib.sha256(Path(path).read_bytes()).hexdigest()==digest for path,digest in real_export_hashes.items())
record_test("pas_de_fuite_vers_reel",True,"DataFrame réel et CSV source inchangés")

# Vérifier périodes triées, mesures finies, effectifs et cibles métier non configurées.
if len(trend_analysis): assert trend_analysis["month_start"].is_monotonic_increasing
if len(group_comparisons): assert (group_comparisons["sample_size"]>0).all()
assert target_check["status"]=="cible métier non configurée"
record_test("contrôles_sorties_réelles",True,{"months":len(trend_analysis),"groups":len(group_comparisons),"target":target_check["status"]})

test_results_df=pd.DataFrame(test_results)
assert test_results_df["passed"].all()
display(test_results_df)

# Visualisations analytiques avec réel et synthétique explicitement séparés.
if len(anomaly_comparison):
    fig,ax=plt.subplots(figsize=(9,4)); ax.plot(anomaly_comparison["period"],anomaly_comparison["score"],marker="o",label="Scores calculables")
    ax.axhline(Z_THRESHOLD,color="red",linestyle="--",label=f"Z +{Z_THRESHOLD}"); ax.axhline(-Z_THRESHOLD,color="red",linestyle="--",label=f"Z -{Z_THRESHOLD}")
    ax.set(title="NYC 311 — scores Z des mois évaluables",xlabel="Mois analysé",ylabel="Z-score"); ax.legend(); plt.xticks(rotation=45); plt.tight_layout(); plt.show()
if stl_result["status"]=="executed":
    stl_decomposition.plot(subplots=True,figsize=(10,8),title="NYC 311 — décomposition STL"); plt.tight_layout(); plt.show()
fig,ax=plt.subplots(figsize=(9,4)); ax.plot(range(len(base)),base,label="Historique synthétique"); ax.scatter([len(base)],[200],color="red",label="Pic injecté")
ax.set(title="TEST SYNTHÉTIQUE — pic artificiel (pas de données NYC 311)",xlabel="Période synthétique",ylabel="Demandes simulées"); ax.legend(); plt.tight_layout(); plt.show()

# Exports enrichis, tout en conservant les quatre chemins historiques.
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
group_path=OUTPUT_DIR/"group_comparisons.csv"; trend_path=OUTPUT_DIR/"trend_analysis.csv"
stl_path=OUTPUT_DIR/"stl_decomposition.csv"; tests_path=OUTPUT_DIR/"test_results.json"
evidence_path=OUTPUT_DIR/"insight_evidence.csv"; comparison_path=OUTPUT_DIR/"anomaly_method_comparison.csv"
group_comparisons.to_csv(group_path,index=False); trend_analysis.to_csv(trend_path,index=False)
if stl_result["status"]=="executed": stl_decomposition.to_csv(stl_path,index=True)
test_results_path_json={"all_passed":bool(test_results_df["passed"].all()),"tests":test_results,
                        "note":"Synthetic tests are in-memory and separate from NYC 311."}
tests_path.write_text(json.dumps(test_results_path_json,ensure_ascii=False,indent=2,default=str),encoding="utf-8")
evidence_registry.to_csv(evidence_path,index=False); anomaly_comparison.to_csv(comparison_path,index=False)
if len(stl_residual_comparison): stl_residual_comparison.to_csv(OUTPUT_DIR/"stl_residual_signals.csv",index=False)
rule_insights.to_csv(insights_path,index=False)
advanced_anomalies.to_csv(anomalies_path,index=False)

enhanced_summary={**summary,"methods_executed":{"zscore":("evaluated" if len(anomaly_comparison) else f"skipped: {len(trend_monthly_volume) if trend_monthly_volume is not None else 0} month(s); need 7"),
    "mad":("evaluated" if len(anomaly_comparison) else f"skipped: {len(trend_monthly_volume) if trend_monthly_volume is not None else 0} month(s); need 7"),"stl":stl_result["status"],
    "stl_reason":stl_result["reason"],"group_comparisons":int(len(group_comparisons)),
    "trend_periods":int(len(trend_analysis)),"targets":target_check["status"]},
    "anomaly_method":{"zscore_threshold":Z_THRESHOLD,"mad_threshold":MAD_THRESHOLD,"min_reference_periods":MIN_REFERENCE_N,
                       "zero_dispersion":"no score and no alert"},
    "anomaly_count":int(len(advanced_anomalies)),"insight_count":int(len(rule_insights)),
    "anomaly_config":{"z_threshold":Z_THRESHOLD,"mad_threshold":MAD_THRESHOLD,"min_reference_n":MIN_REFERENCE_N},
    "rule_version":RULE_VERSION,"quality_report_status":quality_report_status,"raw_quality_status":raw_quality_status,
    "synthetic_tests_all_passed":bool(test_results_df["passed"].all()),"synthetic_test_count":len(test_results),
    "sample_design":None if sample_metadata is None else sample_metadata.get("sample_method"),
    "sample_monthly_counts":None if sample_metadata is None else sample_metadata.get("sample_monthly_counts"),
    "population_monthly_counts":None if sample_metadata is None else sample_metadata.get("population_monthly_counts"),
    "partial_months":sorted(partial_months)}
summary_path.write_text(json.dumps(enhanced_summary,ensure_ascii=False,indent=2,default=str),encoding="utf-8")

report_lines += ["", "## Méthodes statistiques et évaluation", "",
                 f"- Z-score classique : seuil ±{Z_THRESHOLD}; MAD robuste : seuil ±{MAD_THRESHOLD}; référence minimale {MIN_REFERENCE_N} périodes antérieures.",
                 f"- Anomalies réelles : {len(anomaly_comparison)} comparaisons exécutées; {len(advanced_anomalies)} signal(s). {'Historique trop court : détection non exécutée.' if not len(anomaly_comparison) else 'Comparaisons calculées.'}",
                 f"- Tendance persistante : {persistent_volume_trend['status']} ({persistent_volume_trend.get('n_periods',0)} périodes).",
                 f"- STL : {stl_result['status']} — {stl_result['reason'] or 'décomposition calculée'}.",
                 f"- Cible métier : {target_check['status']}. Aucun SLA n’a été inventé.",
                 f"- Tests synthétiques : {len(test_results)} exécutés, tous réussis : {bool(test_results_df['passed'].all())}. Ces résultats ne sont pas des résultats NYC 311.",
                 f"- Rapport qualité externe : {quality_report_status}; source brute : {raw_quality_status}.",
                 f"- Base des volumes mensuels : {monthly_volume['count_basis'].iloc[0] if monthly_volume is not None else 'indisponible'}; mois partiels exclus des tendances : {sorted(partial_months)}. Les comparaisons de catégories/délais utilisent des blocs temporels répartis, non un tirage aléatoire, donc un biais de sélection reste possible.",
                 f"- Insights auditables produits : {len(rule_insights)}; preuves : {len(evidence_registry)}; règle {RULE_VERSION}."]
report_markdown="\n".join(report_lines)
report_path.write_text(report_markdown,encoding="utf-8")

# Vérification des exports réels et du schéma des anomalies, y compris dataframe vide.
assert len(pd.read_csv(insights_path))==len(rule_insights)
assert len(pd.read_csv(anomalies_path))==len(advanced_anomalies)
assert len(pd.read_csv(group_path))==len(group_comparisons)
assert len(pd.read_csv(trend_path))==len(trend_analysis)
assert json.loads(tests_path.read_text(encoding="utf-8"))["all_passed"]
print("Exports renforcés vérifiés :",[str(x) for x in [insights_path,anomalies_path,report_path,summary_path,group_path,trend_path,tests_path,evidence_path,comparison_path]])

## 8. Synthèse locale avec Llama

Les KPI, seuils, anomalies et recommandations opérationnelles sont calculés et exportés par le code déterministe. Llama 3.2 3B ne reçoit que les insights agrégés pour rédiger un court résumé qualitatif, sans chiffres. Si le modèle n’est pas joignable ou produit une sortie douteuse, cette synthèse est omise et le rapport BI déterministe reste disponible.

In [ ]:
# Un seul modèle génératif, local : Llama 3.2 3B via Ollama.
# Le modèle ne reçoit que des insights agrégés, jamais les lignes brutes ni les identifiants.
import os, re, urllib.request

BI_ANALYST_MODEL = "llama3.2:3b"
OLLAMA_BASE_URL = os.getenv("OLLAMA_HOST", "http://localhost:11434").rstrip("/")
llm_summary = None
llm_status = "not_run"
llm_error_type = None

def build_llama_summary(insights_frame):
    evidence_cols = [c for c in ["insight_type", "period", "dimension", "group_value",
                                 "observation", "interpretation", "recommendation", "limitations"]
                     if c in insights_frame.columns]
    if evidence_cols and len(insights_frame):
        facts_frame = insights_frame[evidence_cols].head(12)
        facts = facts_frame.where(pd.notna(facts_frame), "").to_dict("records")
    else:
        facts = [{"constat": "Aucun insight statistique exploitable n’a été produit.",
                  "limite": "Ne pas conclure à l’absence d’anomalie."}]
    prompt = (
        "Rédige en français une synthèse qualitative de deux à quatre phrases à partir des faits JSON fournis. "
        "N’ajoute aucun chiffre, date, cause, événement, recommandation ou fait absent des données. "
        "Conserve les limites et l’incertitude. Si les données sont insuffisantes, dis-le clairement. "
        "Réponds uniquement par le texte, sans JSON, sans titre et sans liste.\n\n"
        + json.dumps(facts, ensure_ascii=False, default=str)
    )
    payload = {"model": BI_ANALYST_MODEL, "stream": False,
               "messages": [{"role": "user", "content": prompt}],
               "options": {"temperature": 0, "num_predict": 220}}
    request = urllib.request.Request(
        OLLAMA_BASE_URL + "/api/chat", data=json.dumps(payload, ensure_ascii=False).encode("utf-8"),
        headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(request, timeout=90) as response:
        result = json.load(response)
    text = str(result.get("message", {}).get("content", "")).strip()
    if not text:
        raise ValueError("empty_model_response")
    if len(text) > 1800:
        raise ValueError("model_response_too_long")
    # Chiffres/KPI du rapport restent entièrement calculés par Python.
    if re.search(r"\d", text):
        raise ValueError("unverified_numeric_claim")
    return text

try:
    llm_summary = build_llama_summary(rule_insights)
    llm_status = "success"
except Exception as exc:
    llm_status = "unavailable_deterministic_report_kept"
    llm_error_type = type(exc).__name__

if llm_summary:
    report_path.write_text(
        report_path.read_text(encoding="utf-8")
        + "\n\n## Synthèse qualitative Llama (sans chiffres)\n\n" + llm_summary + "\n",
        encoding="utf-8")
analysis_summary = json.loads(summary_path.read_text(encoding="utf-8"))
analysis_summary["narrative_model"] = BI_ANALYST_MODEL
analysis_summary["narrative_status"] = llm_status
analysis_summary["narrative_error_type"] = llm_error_type
analysis_summary["narrative_summary"] = llm_summary
summary_path.write_text(json.dumps(analysis_summary, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print("Modèle de synthèse :", BI_ANALYST_MODEL, "| statut :", llm_status,
      "| erreur :", llm_error_type or "aucune")
if llm_summary:
    print(llm_summary)


## Comment exécuter

1. Depuis la racine du projet, exécuter `notebooks/01_etl_kpi.ipynb` afin de produire les données nettoyées.
2. Vérifier qu’Ollama est lancé et que `llama3.2:3b` est installé (`ollama list`).
3. Ouvrir `notebooks/02_bi_analyst.ipynb` dans Jupyter ou VS Code et exécuter toutes les cellules dans l’ordre.
4. Vérifier la création des fichiers dans `outputs/bi_analyst/` et consulter `bi_analyst_report.md`.

Le Notebook 1 n’a pas été modifié. Si Ollama est indisponible, les exports déterministes restent produits et la synthèse Llama est signalée comme indisponible. Cette version Analyste BI reste descriptive : une période courte peut ne permettre ni tendance ni détection statistique, et toute explication causale nécessite des données/contextes complémentaires.